In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import tqdm

from chemlog.classification.charge_classifier import ChargeCategories
from chemlog.preprocessing.chebi_data import ChEBIData

In [ ]:
chebi_classes = {16670: 0, 25676: 0, 46761: 0, 47923: 0, 48030: 0, 48545: 0, 60194: 0, 60334: 0, 60466: 0, 64372: 0, 65061: 0, 90799: 0, 155837: 0, 15841: 0}
all_sums = []
total_time, mean_time = 0, 0
for i in tqdm.tqdm(range(346)):
    with open(os.path.join("results", "pubchem", f"classify_pubchem{i:03d}.json"), "r") as f:
        df = pd.read_json(f)
        #df_sum = df.groupby(["charge_category", "n_amino_acid_residues"])["pubchem_id"].aggregate("count")
        total_time += df["time"].sum()
        mean_time += len(df)
        #all_sums.append(df_sum)
        #for cls in chebi_classes:
        #    chebi_classes[cls] += sum(map(lambda x: cls in x, df["chebi_classes"]))
#all_sums = pd.concat(all_sums)
mean_time = total_time / mean_time
#chebi_classes
total_time, mean_time

In [ ]:
pd.DataFrame(all_sums_dict, index=["charge_category", "n_amino_acid_residues"]).unstack()

In [ ]:
all_sums_agg = pd.DataFrame(all_sums).groupby(["charge_category", "n_amino_acid_residues"]).sum()
all_sums_dict = all_sums_agg.to_dict()["pubchem_id"]
all_sums_ranged = {}
for (charge_category, n_amino_acid_residues), count in all_sums_dict.items():
    if n_amino_acid_residues > 9:
        n_amino_acid_residues = ">9"
    elif n_amino_acid_residues < 2:
        n_amino_acid_residues = "<2"
    else:
        n_amino_acid_residues = str(n_amino_acid_residues)
    if (charge_category, n_amino_acid_residues) in all_sums_ranged:
        all_sums_ranged[(charge_category, n_amino_acid_residues)] += count
    else:
        all_sums_ranged[(charge_category, n_amino_acid_residues)] = count
c, n, p = [], [], []
for (charge_category, n_amino_acid_residues), count in all_sums_ranged.items():
    c.append(charge_category)
    n.append(n_amino_acid_residues)
    p.append(count)
all_sums_ranged = {"charge_category": c, "n_amino_acid_residues": n, "pubchem_id": p}
all_sums_ranged = pd.DataFrame(all_sums_ranged)
all_sums_ranged

In [ ]:
non_peptides = int(all_sums_ranged[all_sums_ranged["n_amino_acid_residues"] == "<2"]["pubchem_id"].sum())
peptides = int(all_sums_ranged[all_sums_ranged["n_amino_acid_residues"] != "<2"]["pubchem_id"].sum())
non_peptides, peptides, peptides / (non_peptides + peptides), peptides + non_peptides

In [ ]:
# plot results by charge category and number of amino acids
import seaborn as sns
import matplotlib.pyplot as plt
sns.barplot(x="n_amino_acid_residues", hue="charge_category", y="pubchem_id", data=all_sums_ranged, legend=True)
plt.yscale("log")
plt.xlabel("Number of amino acid residues")
plt.ylabel("Number of Pubchem entries")
plt.show()

In [ ]:
PEPTIDE_CLASSES = [
    {"chebi_id": 60194, "name": "peptide cation"},
    {"chebi_id": 60334, "name": "peptide anion"},
    {"chebi_id": 60466, "name": "peptide zwitterion"},
    {"chebi_id": 90799, "name": "dipeptide zwitterion"},
    {"chebi_id": 155837, "name": "tripeptide zwitterion"},
    {"chebi_id": 16670, "name": "peptide"},
    {"chebi_id": 25676, "name": "oligopeptide"},
    {"chebi_id": 46761, "name": "dipeptide"},
    {"chebi_id": 47923, "name": "tripeptide"},
    {"chebi_id": 48030, "name": "tetrapeptide"},
    {"chebi_id": 48545, "name": "pentapeptide"},
    {"chebi_id": 15841, "name": "polypeptide"},
    {"chebi_id": 64372, "name": "emericellamide"},
    {"chebi_id": 65061, "name": "2,5-diketopiperazines"}
]
pc_dict = {row["chebi_id"]: row["name"] for row in PEPTIDE_CLASSES}
results_by_chebi_cls = pd.DataFrame.from_dict(chebi_classes, orient="index", columns=["count"])
results_by_chebi_cls["name"] = [pc_dict[cls] for cls in results_by_chebi_cls.index]
results_by_chebi_cls

In [ ]:
# visualise in seaborn
import seaborn as sns
results_by_chebi_cls.sort_values(by="count", ascending=False, inplace=True)
sns.barplot(x="count", y="name", data=results_by_chebi_cls)
plt.xlabel("Number of Pubchem entries")
plt.ylabel("ChEBI class")
plt.xscale("log")